# Gradient Flow and Activation Function Analysis in Neural Networks (From Scratch using NumPy

• Forward propagation

• Backward propagation (manual derivatives)

• Weight updates (Gradient Descent)

• Train on MNIST

# 2-layer Neural Network

Input (784 pixels)

      ↓
Hidden Layer (128 neurons)

      ↓
Output Layer (10 digits)





And manually compute gradients using chain rule.


# Import Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

# Load & Prepare MNIST

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**Normalize**


In [3]:
x_train = x_train / 255.0
x_test = x_test / 255.0

**Flatten**

In [4]:
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

**One-hot encode labels**


In [5]:
def one_hot(y):
    oh = np.zeros((y.size, 10))
    oh[np.arange(y.size), y] = 1
    return oh

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)


# Activation Functions (FROM SCRATCH)
**ReLU**

In [6]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

**Sigmoid**

In [7]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

**Tanh**

In [8]:
def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

**Leaky ReLU**

In [9]:
def leaky_relu(x):
    return np.where(x > 0, x, 0.01*x)

def leaky_relu_derivative(x):
    return np.where(x > 0, 1, 0.01)

# Choose Activation

In [11]:
ACTIVATION = "relu"
ACTIVATION = "sigmoid"
ACTIVATION = "tanh"
ACTIVATION = "relu"
ACTIVATION = "leaky"

# Activation Selector

In [25]:
def activate(x):
    if ACTIVATION == "relu":
        return relu(x)
    elif ACTIVATION == "sigmoid":
        return sigmoid(x)
    elif ACTIVATION == "tanh":
        return tanh(x)
    elif ACTIVATION == "leaky":
        return leaky_relu(x)

def activate_derivative(x):
    if ACTIVATION == "relu":
        return relu_derivative(x)
    elif ACTIVATION == "sigmoid":
        return sigmoid_derivative(x)
    elif ACTIVATION == "tanh":
        return tanh_derivative(x)
    elif ACTIVATION == "leaky":
        return leaky_relu_derivative(x)

# Softmax

In [26]:
def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

# Initialize Weights

I used **He initialization** (important for ReLU)

In [27]:
np.random.seed(42)

W1 = np.random.randn(784, 128) * np.sqrt(2/784)
b1 = np.zeros((1,128))

W2 = np.random.randn(128, 10) * np.sqrt(2/128)
b2 = np.zeros((1,10))

# Forward Propagation

In [28]:
def forward(x):

    # layer 1
    Z1 = x @ W1 + b1
    A1 = activate(Z1)

    # output layer
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)

    cache = (Z1, A1, Z2, A2)
    return A2, cache

# Loss Function (Cross Entropy)

In [29]:
def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    loss = -np.sum(y_true * np.log(y_pred + 1e-9)) / m
    return loss

# Backpropagation

This is where deep learning actually happens

In [30]:
learning_rate = 0.01

def backward(x, y, cache):

    global W1, b1, W2, b2

    Z1, A1, Z2, A2 = cache
    m = x.shape[0]

    # Output layer
    dZ2 = A2 - y
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Hidden layer
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * activate_derivative(Z1)
    dW1 = (x.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    # Gradient Descent update
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

# Training Loop

In [31]:
epochs = 20
batch_size = 128

for epoch in range(epochs):

    for i in range(0, x_train.shape[0], batch_size):

        x_batch = x_train[i:i+batch_size]
        y_batch = y_train_oh[i:i+batch_size]

        y_pred, cache = forward(x_batch)
        loss = compute_loss(y_batch, y_pred)
        backward(x_batch, y_batch, cache)

    print(f"Epoch {epoch+1} | Loss: {loss:.4f}")


Epoch 1 | Loss: 0.7745
Epoch 2 | Loss: 0.5419
Epoch 3 | Loss: 0.4596
Epoch 4 | Loss: 0.4168
Epoch 5 | Loss: 0.3898
Epoch 6 | Loss: 0.3709
Epoch 7 | Loss: 0.3566
Epoch 8 | Loss: 0.3452
Epoch 9 | Loss: 0.3359
Epoch 10 | Loss: 0.3282
Epoch 11 | Loss: 0.3213
Epoch 12 | Loss: 0.3149
Epoch 13 | Loss: 0.3092
Epoch 14 | Loss: 0.3039
Epoch 15 | Loss: 0.2990
Epoch 16 | Loss: 0.2944
Epoch 17 | Loss: 0.2900
Epoch 18 | Loss: 0.2859
Epoch 19 | Loss: 0.2819
Epoch 20 | Loss: 0.2781


# Testing Accuracy

In [32]:
def accuracy(x, y):

    pred, _ = forward(x)
    predicted_labels = np.argmax(pred, axis=1)

    return np.mean(predicted_labels == y)

print("Test Accuracy:", accuracy(x_test, y_test))

Test Accuracy: 0.9383


**WITHOUT any deep learning library**

| Activation | What Happens                       |
| ---------- | ---------------------------------- |
| Sigmoid    | Slow learning (vanishing gradient) |
| Tanh       | Better but still slow              |
| ReLU       | Fast training                      |
| Leaky ReLU | Most stable                        |
